Load dataset with memory optimization

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Loading merged dataset...")
# Read the CSV, but specify dtypes to save memory
# We'll load it normally first; we can convert types later
df = pd.read_csv('merged_cve_dataset.csv', low_memory=False)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

Loading merged dataset...
Loaded 2,830,743 rows, 79 columns
Memory usage: 1.81 GB


Replace infinite values & drop NaNs

In [2]:
import numpy as np

print("Original shape:", df.shape)

# Replace infinities with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# Check missing values per column
missing = df.isnull().sum()
print(f"Missing values per column (top 5):\n{missing[missing>0].head(5)}")

# Drop rows with any missing values
df = df.dropna()
print(f"After dropping NaNs: {len(df):,} rows")

Original shape: (2830743, 79)
Missing values per column (top 5):
Flow Bytes/s       2867
 Flow Packets/s    2867
dtype: int64
After dropping NaNs: 2,827,876 rows


Encode Labels (Multi-class)

In [6]:
# Clean column names - remove leading/trailing spaces
df.columns = df.columns.str.strip()

# Verify the cleaned column names
print("Cleaned column names (first 10):")
print(df.columns[:10].tolist())
print("\nLabel column now:", 'Label' in df.columns)
print("Label column exists:", 'Label' in df.columns)

# Now inspect unique labels
print("\nUnique labels:\n", df['Label'].value_counts())

# Create label mapping
labels = df['Label'].unique()
label_to_int = {label: i for i, label in enumerate(sorted(labels))}
print("\nLabel mapping:")
for k, v in label_to_int.items():
    print(f"  {k} -> {v}")

# Apply mapping
df['Label_encoded'] = df['Label'].map(label_to_int)

# Save mapping for later (when predicting)
import json
with open('label_mapping.json', 'w') as f:
    json.dump(label_to_int, f)

print("\n✅ Label encoding complete. New column 'Label_encoded' added.")
print("\nLabel distribution (encoded):")
print(df['Label_encoded'].value_counts())

# Show sample
print("\nSample of data with encoded labels:")
print(df[['Destination Port', 'Flow Duration', 'Label', 'Label_encoded']].head())

Cleaned column names (first 10):
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']

Label column now: True
Label column exists: True

Unique labels:
 Label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Label mapping:
  BENIGN -> 0
  Bot -> 1
  DDoS ->

In [5]:
# Check all column names
print("All columns in dataframe:")
print(df.columns.tolist())

# Check for any column that might contain labels
print("\nColumns that might contain labels:")
label_cols = [col for col in df.columns if 'label' in col.lower() or 'attack' in col.lower() or 'class' in col.lower()]
print(label_cols)

# Check the first few rows to see the data
print("\nFirst 5 rows:")
print(df.head())

All columns in dataframe:
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' AC

In [8]:
# Create label mapping
labels = df['Label'].unique()
label_to_int = {label: i for i, label in enumerate(sorted(labels))}
print("Label mapping:")
for k, v in label_to_int.items():
    print(f"  {k} -> {v}")

# Apply mapping
df['Label_encoded'] = df['Label'].map(label_to_int)

# Save mapping for later
import json
with open('label_mapping.json', 'w') as f:
    json.dump(label_to_int, f)

print("Label encoding complete. New column 'Label_encoded' added.")

Label mapping:
  BENIGN -> 0
  Bot -> 1
  DDoS -> 2
  DoS GoldenEye -> 3
  DoS Hulk -> 4
  DoS Slowhttptest -> 5
  DoS slowloris -> 6
  FTP-Patator -> 7
  Heartbleed -> 8
  Infiltration -> 9
  PortScan -> 10
  SSH-Patator -> 11
  Web Attack � Brute Force -> 12
  Web Attack � Sql Injection -> 13
  Web Attack � XSS -> 14
Label encoding complete. New column 'Label_encoded' added.


Encode Labels

Prepare the Data

In [10]:
# Separate features (all except Label and Label_encoded)
X = df.drop(columns=['Label', 'Label_encoded'])
y = df['Label_encoded']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of classes: {y.nunique()}")

Features shape: (2827876, 78)
Target shape: (2827876,)
Number of classes: 15


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set size: {X_train.shape[0]:,} rows")
print(f"Test set size: {X_test.shape[0]:,} rows")

Training set size: 2,262,300 rows
Test set size: 565,576 rows


Train Random Forest on Full Dataset

In [13]:
from sklearn.ensemble import RandomForestClassifier
import joblib
import time

print("Training Random Forest on 2.8M rows...")
start = time.time()

rf = RandomForestClassifier(
    n_estimators=50,        # Reduced from default 100 for speed
    max_depth=20,           # Limit depth to avoid overfitting
    n_jobs=-1,              # Use all CPU cores
    random_state=42,
    verbose=1               # Shows progress (one dot per tree)
)

rf.fit(X_train, y_train)

print(f"Training completed in {time.time() - start:.2f} seconds")

Training Random Forest on 2.8M rows...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   57.6s


Training completed in 78.67 seconds


[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:  1.3min finished


📊 Evaluate and Save the Model

In [14]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

# Save model
joblib.dump(rf, 'random_forest_full.pkl')
print("Model saved as random_forest_full.pkl")

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.8s
[Parallel(n_jobs=8)]: Done  50 out of  50 | elapsed:    0.9s finished


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    454265
           1       0.90      0.49      0.64       391
           2       1.00      1.00      1.00     25605
           3       1.00      0.99      0.99      2059
           4       1.00      1.00      1.00     46025
           5       0.99      0.99      0.99      1100
           6       1.00      1.00      1.00      1159
           7       1.00      1.00      1.00      1587
           8       1.00      1.00      1.00         2
           9       1.00      0.57      0.73         7
          10       0.99      1.00      1.00     31761
          11       1.00      1.00      1.00      1180
          12       0.70      0.96      0.81       301
          13       0.00      0.00      0.00         4
          14       0.47      0.05      0.10       130

    accuracy                           1.00    565576
   macro avg       0.87      0.80      0.82    565576
weighted avg       1.00   

In [15]:
feature_names = X.columns.tolist()
with open('feature_names.txt', 'w') as f:
    for name in feature_names:
        f.write(f"{name}\n")
print("Feature names saved to feature_names.txt")

Feature names saved to feature_names.txt
